# Task 1 — Trích xuất dữ liệu Shot từ StatsBomb (FIFA World Cup 2018 + 2022)

**Mục tiêu:** dataset gốc quá lớn (nhiều file JSON lồng nhau), nên bước này chỉ trích các field cần thiết cho bài toán **Shot Zone / Shot Quality Clustering** (K-Means → KNN) từ toàn bộ trận đấu của **FIFA World Cup 2018 và 2022**, rồi ghi ra 1 file `.csv` phẳng để các Task sau (thống kê mô tả, EDA, preprocessing, modeling) dùng lại mà không cần đụng tới JSON gốc nữa.

**Phạm vi dữ liệu:** cố định **FIFA World Cup**, `competition_id = 43`, lấy **cả 2 mùa full 64 trận**:
- `season_id = 3` → **World Cup 2018**
- `season_id = 106` → **World Cup 2022**

(Đây là 2 mùa duy nhất của World Cup trong StatsBomb open-data có đủ **full 64 trận**; các kỳ World Cup cũ hơn 1958–1990 chỉ có vài trận lẻ nên không dùng ở đây.)

**Nguồn dữ liệu:** Kaggle Dataset **`saurabhshahane/statsbomb-football-data`** — khi bạn add dataset này vào notebook, Kaggle sẽ mount vào `/kaggle/input/statsbomb-football-data/`.

**Lưu ý quan trọng:** mình không có quyền truy cập trực tiếp vào Kaggle để xem cấu trúc thư mục chính xác bên trong dataset đó, nên notebook này có 1 bước **auto-discovery** — tự quét toàn bộ thư mục input để tìm `competitions.json`, các file `matches/{competition_id}/{season_id}.json` và `events/{match_id}.json`, bất kể chúng nằm sâu bao nhiêu cấp thư mục con (có "data/" bọc ngoài hay không, có thêm 1 lớp tên dataset hay không...). Nhờ vậy notebook vẫn chạy đúng mà không cần bạn phải tự dò đường dẫn thủ công. Nếu cấu trúc dataset quá khác biệt, cell auto-discovery sẽ in ra thông báo cụ thể để bạn biết chỗ cần sửa.


## 1.0 — Import thư viện

In [1]:
import json
import math
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


## 1.0 (tiếp) — Config

In [2]:
# ---------------- Config ----------------

# Cố định FIFA World Cup
COMPETITION_ID = 43

# World Cup 2018 (season_id=3) và World Cup 2022 (season_id=106) - cả 2 đều full 64 trận
SEASON_IDS = [3, 106]

# Giới hạn số trận xử lý MỖI mùa (để test nhanh). Đặt None để chạy toàn bộ 64+64=128 trận.
MAX_MATCHES_PER_SEASON = None   # đổi thành vd. 5 nếu muốn test nhanh trước

# Đường dẫn Kaggle Dataset input (đúng slug bạn đã add: saurabhshahane/statsbomb-football-data)
KAGGLE_INPUT_DIR = Path("/kaggle/input/datasets/saurabhshahane/statsbomb-football-data/data")

# Thư mục output trên Kaggle
WORK_DIR = Path("/kaggle/working")
OUTPUT_CSV = WORK_DIR / "shots_worldcup_2018_2022_raw.csv"

# Tọa độ khung thành (hệ tọa độ sân chuẩn StatsBomb 120 x 80)
GOAL_X, GOAL_Y = 120.0, 40.0


## 1.1 — Auto-discovery: tự dò cấu trúc thư mục dataset

Quét 1 lần toàn bộ `KAGGLE_INPUT_DIR`, lập chỉ mục (index) vị trí thật của:
- `competitions.json`
- từng file `matches/{competition_id}/{season_id}.json`
- từng file `events/{match_id}.json`

Cách này không phụ thuộc vào việc dataset có bọc thêm thư mục `data/` hay tên dataset ở ngoài hay không.

In [3]:
def discover_paths(base_dir: Path):
    competitions_path = None
    matches_index = {}   # (competition_id, season_id) -> Path
    events_index = {}    # match_id (str) -> Path

    all_json_files = list(base_dir.rglob("*.json"))

    for p in all_json_files:
        parts = p.parts

        if p.name == "competitions.json" and competitions_path is None:
            competitions_path = p
            continue

        # matches/{competition_id}/{season_id}.json
        if len(parts) >= 3 and parts[-3].lower() == "matches":
            competition_id, season_id = parts[-2], p.stem
            if competition_id.isdigit() and season_id.isdigit():
                matches_index[(int(competition_id), int(season_id))] = p
                continue

        # events/{match_id}.json
        if len(parts) >= 2 and parts[-2].lower() == "events":
            match_id = p.stem
            if match_id.isdigit():
                events_index[match_id] = p

    return competitions_path, matches_index, events_index, len(all_json_files)


assert KAGGLE_INPUT_DIR.exists(), (
    f"Không tìm thấy thư mục {KAGGLE_INPUT_DIR}. "
    "Hãy kiểm tra lại bạn đã add đúng dataset 'saurabhshahane/statsbomb-football-data' "
    "vào notebook chưa (nút '+ Add Input' bên phải), và đúng slug tên dataset."
)

competitions_path, matches_index, events_index, n_json_files = discover_paths(KAGGLE_INPUT_DIR)

print(f"Tổng số file .json quét được trong dataset: {n_json_files}")
print(f"competitions.json  : {competitions_path}")
print(f"Số entry matches đã index : {len(matches_index)}")
print(f"Số entry events đã index  : {len(events_index)}")

assert competitions_path is not None, "Không tìm thấy competitions.json trong dataset — kiểm tra lại cấu trúc thư mục."


Tổng số file .json quét được trong dataset: 8977
competitions.json  : /kaggle/input/datasets/saurabhshahane/statsbomb-football-data/data/competitions.json
Số entry matches đã index : 80
Số entry events đã index  : 4235


In [4]:
# Kiểm tra nhanh: có đủ 2 season World Cup mình cần không
for season_id in SEASON_IDS:
    key = (COMPETITION_ID, season_id)
    status = "OK" if key in matches_index else "KHÔNG TÌM THẤY"
    print(f"competition_id={COMPETITION_ID}, season_id={season_id} -> {status}")


competition_id=43, season_id=3 -> OK
competition_id=43, season_id=106 -> OK


## 1.2 — Hàm nạp dữ liệu qua index đã dò được

In [5]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_match_ids(competition_id: int, season_id: int):
    matches_path = matches_index.get((competition_id, season_id))
    if matches_path is None:
        raise FileNotFoundError(
            f"Không tìm thấy file matches cho competition_id={competition_id}, season_id={season_id}"
        )
    matches = load_json(matches_path)
    return [m["match_id"] for m in matches]


def get_events(match_id: int):
    events_path = events_index.get(str(match_id))
    if events_path is None:
        raise FileNotFoundError(f"Không tìm thấy file events cho match_id={match_id}")
    return load_json(events_path)


## 1.3 — Trích field cần thiết từ mỗi shot event + tính feature dẫn xuất

Field chia làm 3 nhóm (đúng theo thiết kế đã thống nhất):

| Nhóm | Field |
|---|---|
| **Định danh** | `event_id`, `match_id`, `season_id`, `team_id/name`, `player_id/name`, `period`, `minute`, `second`, `play_pattern` |
| **Feature cho model (X)** | `location_x/y`, `distance_to_goal`, `angle_to_goal`, `under_pressure`, `body_part`, `technique`, `shot_type`, `first_time`, `aerial_won`, `open_goal`, `n_teammates_in_frame`, `n_opponents_in_frame`, `keeper_x/y`, `end_location_x/y` |
| **Field giấu để validate cụm sau này (KHÔNG đưa vào model)** | `outcome`, `statsbomb_xg` |

`distance_to_goal` và `angle_to_goal` được tính sẵn từ `location` và tọa độ khung thành `(120, 40)`.

In [6]:
def compute_distance_angle(x, y):
    dx = GOAL_X - x
    dy = GOAL_Y - y
    distance = math.hypot(dx, dy)
    angle = math.atan2(dy, dx)
    return distance, angle


def extract_shots_from_match(match_id: int, season_id: int):
    events = get_events(match_id)
    rows = []

    for e in events:
        if e.get("type", {}).get("name") != "Shot":
            continue

        shot = e.get("shot", {})
        loc = e.get("location", [None, None])
        x, y = loc[0], loc[1]

        distance_to_goal, angle_to_goal = (None, None)
        if x is not None and y is not None:
            distance_to_goal, angle_to_goal = compute_distance_angle(x, y)

        freeze_frame = shot.get("freeze_frame", []) or []
        n_teammates_in_frame = sum(1 for p in freeze_frame if p.get("teammate") is True)
        n_opponents_in_frame = sum(1 for p in freeze_frame if p.get("teammate") is False)

        keeper_entries = [
            p for p in freeze_frame
            if p.get("teammate") is False and p.get("position", {}).get("name") == "Goalkeeper"
        ]
        keeper_x = keeper_entries[0]["location"][0] if keeper_entries else None
        keeper_y = keeper_entries[0]["location"][1] if keeper_entries else None

        end_loc = shot.get("end_location", [None, None, None]) or [None, None, None]

        row = {
            # --- Định danh ---
            "event_id": e.get("id"),
            "match_id": match_id,
            "season_id": season_id,
            "team_id": e.get("team", {}).get("id"),
            "team_name": e.get("team", {}).get("name"),
            "player_id": e.get("player", {}).get("id"),
            "player_name": e.get("player", {}).get("name"),
            "period": e.get("period"),
            "minute": e.get("minute"),
            "second": e.get("second"),
            "play_pattern": e.get("play_pattern", {}).get("name"),

            # --- Feature cho model (X) ---
            "location_x": x,
            "location_y": y,
            "distance_to_goal": distance_to_goal,
            "angle_to_goal": angle_to_goal,
            "under_pressure": bool(e.get("under_pressure", False)),
            "body_part": shot.get("body_part", {}).get("name"),
            "technique": shot.get("technique", {}).get("name"),
            "shot_type": shot.get("type", {}).get("name"),
            "first_time": bool(shot.get("first_time", False)),
            "aerial_won": bool(shot.get("aerial_won", False)),
            "open_goal": bool(shot.get("open_goal", False)),
            "n_teammates_in_frame": n_teammates_in_frame,
            "n_opponents_in_frame": n_opponents_in_frame,
            "keeper_x": keeper_x,
            "keeper_y": keeper_y,
            "end_location_x": end_loc[0],
            "end_location_y": end_loc[1],

            # --- Field giấu để validate cụm sau này (KHÔNG dùng làm feature) ---
            "outcome": shot.get("outcome", {}).get("name"),
            "statsbomb_xg": shot.get("statsbomb_xg"),
        }
        rows.append(row)

    return rows


## 1.4 — Chạy trích xuất toàn bộ trận đấu của World Cup 2018 + 2022

In [7]:
all_rows = []
failed_matches = []

for season_id in SEASON_IDS:
    match_ids = get_match_ids(COMPETITION_ID, season_id)
    if MAX_MATCHES_PER_SEASON is not None:
        match_ids = match_ids[:MAX_MATCHES_PER_SEASON]

    print(f"Season {season_id}: {len(match_ids)} trận sẽ được xử lý")

    for match_id in tqdm(match_ids, desc=f"Season {season_id}"):
        try:
            rows = extract_shots_from_match(match_id, season_id)
            all_rows.extend(rows)
        except Exception as ex:
            print(f"[WARN] Lỗi ở match_id={match_id}: {ex}")
            failed_matches.append(match_id)

print(f"\nTổng số shot trích xuất được: {len(all_rows)}")
if failed_matches:
    print(f"Số trận bị lỗi (bỏ qua): {len(failed_matches)} -> {failed_matches}")


Season 3: 64 trận sẽ được xử lý


Season 3:   0%|          | 0/64 [00:00<?, ?it/s]

Season 106: 64 trận sẽ được xử lý


Season 106:   0%|          | 0/64 [00:00<?, ?it/s]


Tổng số shot trích xuất được: 3200


In [8]:
df = pd.DataFrame(all_rows)
df.shape


(3200, 30)

## 1.5 — Kiểm tra nhanh trước khi lưu (sanity check)

- Số dòng / số cột có hợp lý không (dự kiến ~1300-1400 shot cho 128 trận, dựa trên tỷ lệ trung bình ~0.79% event là Shot).
- Số shot theo từng season (2018 vs 2022) có cân đối không (đều 64 trận nên số shot nên xấp xỉ nhau).
- Kiểu dữ liệu từng cột, tỷ lệ missing.
- Encoding tên cầu thủ có dấu đọc lại đúng không.

In [9]:
print("Shape:", df.shape)
df.dtypes


Shape: (3200, 30)


event_id                 object
match_id                  int64
season_id                 int64
team_id                   int64
team_name                object
player_id                 int64
player_name              object
period                    int64
minute                    int64
second                    int64
play_pattern             object
location_x              float64
location_y              float64
distance_to_goal        float64
angle_to_goal           float64
under_pressure             bool
body_part                object
technique                object
shot_type                object
first_time                 bool
aerial_won                 bool
open_goal                  bool
n_teammates_in_frame      int64
n_opponents_in_frame      int64
keeper_x                float64
keeper_y                float64
end_location_x          float64
end_location_y          float64
outcome                  object
statsbomb_xg            float64
dtype: object

In [10]:
df["season_id"].value_counts()


season_id
3      1706
106    1494
Name: count, dtype: int64

In [11]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)


keeper_x    125
keeper_y    125
dtype: int64

In [12]:
print(df["outcome"].value_counts())
print("\nstatsbomb_xg describe:")
print(df["statsbomb_xg"].describe())


outcome
Off T               1020
Blocked              836
Saved                687
Goal                 378
Wayward              214
Post                  60
Saved to Post          3
Saved Off Target       2
Name: count, dtype: int64

statsbomb_xg describe:
count    3200.000000
mean        0.122290
std         0.180206
min         0.000180
25%         0.028790
50%         0.059033
75%         0.116540
max         0.991189
Name: statsbomb_xg, dtype: float64


In [13]:
sample_accented = df[df["player_name"].str.contains("í|é|ó|ñ|ü|ã", regex=True, na=False)]["player_name"].unique()
sample_accented[:10]


array(['José Paulo Bezzera Maciel Júnior', 'Aníbal Cesis Godoy',
       'Felipe Abdiel Baloy Ramírez', 'Blas Antonio Miguel Pérez Ortega',
       'José Luis Rodríguez Francis', 'Salif Sané', 'Sadio Mané',
       'Ricardo Iván Rodríguez Araya', 'Héctor Miguel Herrera López',
       'Héctor Alfredo Moreno Herrera'], dtype=object)

## 1.6 — Ghi ra file CSV

Dùng encoding `utf-8-sig` để Excel/pandas đọc lại tên cầu thủ có dấu không bị lỗi font.

In [14]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Đã lưu: {OUTPUT_CSV}  ({df.shape[0]} dòng, {df.shape[1]} cột)")


Đã lưu: /kaggle/working/shots_worldcup_2018_2022_raw.csv  (3200 dòng, 30 cột)


In [15]:
check_df = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
check_df.head(5)


,event_id,match_id,season_id,team_id,team_name,player_id,player_name,period,minute,second,...,aerial_won,open_goal,n_teammates_in_frame,n_opponents_in_frame,keeper_x,keeper_y,end_location_x,end_location_y,outcome,statsbomb_xg
0,9d2b23a7-0fde-45f1-8b6e-58fc9da4aab5,8650,3,782,Belgium,3089,Kevin De Bruyne,1,1,34,...,False,False,4,8,118.0,41.0,120.0,34.8,Off T,0.020461
1,ff6fee87-0f6b-461e-a5fd-e5032aa19564,8650,3,781,Brazil,3295,Thiago Emiliano da Silva,1,7,12,...,False,False,4,8,118.0,38.0,120.0,35.8,Post,0.274021
2,ebeaf37c-a1be-46a4-b3bb-ae4779458734,8650,3,782,Belgium,3621,Eden Hazard,1,7,32,...,False,False,2,8,119.0,39.0,109.0,31.0,Blocked,0.065900
3,0f142c55-8a4a-4af1-b57b-e8eba2491c4b,8650,3,782,Belgium,4831,Nacer Chadli,1,7,35,...,False,False,4,10,119.0,41.0,120.0,34.4,Off T,0.022908
4,06ffa199-3887-4696-b954-5ba6376cbce6,8650,3,781,Brazil,5542,José Paulo Bezzera Maciel Júnior,1,9,12,...,False,False,3,9,118.0,40.0,111.0,45.0,Blocked,0.174216


## Nếu auto-discovery báo lỗi / không tìm thấy dataset

1. Chạy thử lệnh sau trong 1 cell riêng để tự xem cấu trúc thật của dataset trên máy bạn, rồi báo lại để chỉnh code:
```python
for p in list(KAGGLE_INPUT_DIR.rglob("*"))[:30]:
    print(p)
```
2. Kiểm tra lại đúng tên dataset đã add trong phần **+ Add Input** ở góc phải notebook (đúng slug `statsbomb-football-data`, vì mount path phụ thuộc chính xác vào slug này).
3. Nếu dataset trên Kaggle có tên file/thư mục khác quy ước gốc của StatsBomb (ví dụ đổi tên `events` thành `event`), sửa lại điều kiện so khớp `"events"` / `"matches"` trong hàm `discover_paths` ở cell 1.1 cho khớp.

## Kết quả Task 1

File `/kaggle/working/shots_worldcup_2018_2022_raw.csv` chứa toàn bộ shot event của FIFA World Cup 2018 và 2022 (128 trận), đã tách sẵn 3 nhóm field (định danh / feature cho model / field giấu để validate). Đây là input cho **Task 2 — Thống kê mô tả** và **Task 3 — EDA** ở bước tiếp theo.
